In [1]:
# ======================
# Step 1. Import libraries
# ======================
import pandas as pd
import os

base_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"
output_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"
os.makedirs(output_path, exist_ok=True)

# ======================
# Step 2. Load source data
# ======================
mimic_dx = pd.read_csv(os.path.join(base_path, "hosp_diagnoses_icd.csv"), dtype=str)
mimic_dict = pd.read_csv(os.path.join(base_path, "hosp_d_icd_diagnoses.csv"), dtype=str)

print("hosp_diagnoses_icd:", mimic_dx.shape)
print("hosp_d_icd_diagnoses:", mimic_dict.shape)

# ======================
# Step 3. Merge with dictionary to add long_title
# ======================
dx_merged = mimic_dx.merge(
    mimic_dict,
    on=["icd_code", "icd_version"],
    how="left"
)

print("Merged diagnoses:", dx_merged.shape)

# ======================
# Step 4. Split ICD-9 and ICD-10
# ======================
dx_merged["dx_code_icd9"] = dx_merged.apply(
    lambda x: x["icd_code"] if x["icd_version"] == "9" else None,
    axis=1
)

dx_merged["dx_code_icd10"] = dx_merged.apply(
    lambda x: x["icd_code"] if x["icd_version"] == "10" else None,
    axis=1
)

# ======================
# Step 5. Build final DataFrame
# ======================
diagnosis_final = pd.DataFrame({
    "pat_id": dx_merged["subject_id"],
    "csn": dx_merged["hadm_id"],
    "dx_code_icd9": dx_merged["dx_code_icd9"],
    "dx_code_icd10": dx_merged["dx_code_icd10"],
    "dx_time_date": "NOT AVAILABLE",
    "long_title": dx_merged["long_title"].fillna("UNKNOWN")
})

# drop duplicates if needed
diagnosis_final = diagnosis_final.drop_duplicates()

print("✅ Final DIAGNOSIS shape:", diagnosis_final.shape)
print(diagnosis_final.head(10))

# ======================
# Step 6. Save output
# ======================
out_file = os.path.join(output_path, "DIAGNOSIS.csv")
diagnosis_final.to_csv(out_file, index=False)
print(f"🎉 DIAGNOSIS file saved to {out_file}")


hosp_diagnoses_icd: (6364488, 5)
hosp_d_icd_diagnoses: (112107, 3)
Merged diagnoses: (6364488, 6)
✅ Final DIAGNOSIS shape: (6364100, 6)
     pat_id       csn dx_code_icd9 dx_code_icd10   dx_time_date  \
0  10000032  22595853      5723             None  NOT AVAILABLE   
1  10000032  22595853      78959            None  NOT AVAILABLE   
2  10000032  22595853      5715             None  NOT AVAILABLE   
3  10000032  22595853      07070            None  NOT AVAILABLE   
4  10000032  22595853      496              None  NOT AVAILABLE   
5  10000032  22595853      29680            None  NOT AVAILABLE   
6  10000032  22595853      30981            None  NOT AVAILABLE   
7  10000032  22595853      V1582            None  NOT AVAILABLE   
8  10000032  22841357      07071            None  NOT AVAILABLE   
9  10000032  22841357      78959            None  NOT AVAILABLE   

                                          long_title  
0                                Portal hypertension  
1               